# Get to Know a Dataset: DE Africa Sentinel-3 LST

This notebook serves as a guided tour of the [Digital Earth Africa Sentinel-3 SLSTR Level-2 LST](https://registry.opendata.aws/[REGISTRY_YAML_NAME]) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).



### Dataset Organization & S3 Prefix Structure Overview
***

At the top level of our S3 buckets, we follow a **consistent, product-centric prefix structure** that supports scalable, cloud-native access to **Sentinel-3 SLSTR Level-2 LST products stored as Cloud-Optimised GeoTIFFs (COGs)**.

##### Dataset organisation and key S3 prefix structure
1. **Top-level S3 bucket (per product)**

   The variable is stored in its own dedicated S3 bucket, for example:

   * `s3://deafrica-sentinel-3-slstr-l2-lst/Sentinel-3/SLSTR/SL_2_LST/`

   This separation simplifies data discovery, access management, and scaling as data volumes grow.

2. **Mission prefix**

   ```
   deafrica-sentinel-3-slstr-l2-lst/
   ```

   Identifies the satellite mission and allows future missions to be added without restructuring existing paths.

3. **Instrument prefix**

   ```
   SLSTR/
   ```

   Specifies the sensor producing the observations and keeps all products organised consistently.

4. **Processing level and product prefix**

   Each Level-2 product is stored in a dedicated directory, for example:

   * `SL_2_LST` – Sentinel-3 SLSTR Level-2 LST

   This ensures a clear one-to-one mapping between scientific products and directory paths.

5. **Temporal partitioning**

   Products are partitioned by time using:

   ```
   YYYY/MM/DD/
   ```

   or, where appropriate:

   ```
   YYYY/MM/
   ```

   Example:

   ```
   s3://deafrica-sentinel-3-slstr-l2-lst/Sentinel-3/SLSTR/SL_2_LST/
   ```

   Temporal partitioning enables efficient S3 prefix filtering and time-based subsetting in cloud workflows.

6. **Cloud-Optimised GeoTIFFs (COGs)**

   Within each temporal directory are individual **Cloud-Optimised GeoTIFF (`.tif`) files**, where:

   * each file represents a single Sentinel-3 SLSTR Level-2 LST
   * internal tiling and overviews enable HTTP range requests
   * files can be streamed directly into GIS tools, xarray/rioxarray, and Dask without full downloads


Full documentation for this dataset can be found at: [https://docs.digitalearthafrica.org/en/latest/sandbox/notebooks/Datasets/Sentinel_3_Land_Surface_Temperature.html](https://docs.digitalearthafrica.org/en/latest/sandbox/notebooks/Datasets/Sentinel_3_Land_Surface_Temperature.html)




#### Install and import packages

Depending on your environment, you may need to install some packages before importing them. The installation commands in the cell below are effective in Sagemaker Studio Lab (with kernel set to `sagemaker-distribution:Python`) and Google Colab.

If you are running the notebook for the first time, you can remove the `#`s and run the cell below. Often it is more aesthetic to add the `#`s back in after the cell has run because the outputs take a lot of space!

You may see some dependency resolver errors in the output, though this should not affect performance and did not during testing in Colab and SageMaker.

In [1]:
# This notebook requires the following additional libraries
# (Please install using the preferred method for your environment by uncommenting the appropriate lines below):

# !pip install pystac==1.10.0 --force-reinstall
# !pip install odc-stac
# !pip install osmnx
# !pip install pystac_client
# !pip install odc-algo
# !pip install matplotlib

In [2]:
# Import the libraries required for this notebook
# Built-ins
import numpy as np
import xarray as xr
import pandas as pd

# Installed libraries
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import odc.algo
from pystac_client import Client
from odc.stac import configure_rio, stac_load

### Data Formats, Content Types & Practical Usage Guidance

***

Our dataset is stored using **cloud-native geospatial formats**, chosen to support scalable access, analysis, and visualisation directly from S3 without full file downloads.

#### Data formats present in the dataset

##### 1. Cloud-Optimised GeoTIFF (COG) — `.tif`

**What is stored**

* **Sentinel-3 SLSTR Level-2 LST** 
* Each file represents a **single overpass or temporally aggregated product**
* Raster grids with georeferencing, metadata, and quality information

**Why this format is used**

* Internally tiled with overviews
* Supports HTTP range requests
* Optimised for direct cloud access from S3

**How we work with it**

* Stream directly from S3 using `rasterio`, `rioxarray`, or `xarray`
* Subset spatially and temporally without downloading full files
* Visualise instantly in GIS tools (QGIS, ArcGIS)

**Advice**

* Always treat these as *remote datasets*—avoid downloading unless necessary
* Use windowed reads and bounding boxes for performance
* Prefer `rioxarray.open_rasterio()` or STAC-based loading for analysis



##### 2. STAC metadata (JSON)

**What is stored**

* Collection-level and item-level metadata exposed via the **Digital Earth Africa STAC catalogue**
* Spatial extent, temporal coverage, projection, and asset links pointing to the COGs in S3

**Why this format is used**

* Enables structured discovery and filtering of large datasets
* Decouples metadata from physical storage layout

**How we work with it**

* Query using `pystac_client`
* Filter by bounding box, time range, and product type
* Use STAC assets to access COGs programmatically

**Advice**

* Use STAC as the *entry point* rather than hard-coding S3 paths
* Let STAC handle discovery; let raster tools handle reading

This approach keeps workflows **fast, scalable, and cloud-efficient**, even when working with large time series of Sentinel-3 SLSTR Level-2 LST products.

----

### Downloading and Loading Data from the Dataset

We define the location of our dataset, configure access to AWS S3, and prepare the environment for cloud-native data access

We start by configuring Rasterio/GDAL for unsigned (public) S3 access and explicitly point to the Africa (Cape Town) AWS region, where the data are hosted.
This configuration ensures that Cloud-Optimised GeoTIFFs can be accessed efficiently via HTTP range requests without downloading entire files.

In [3]:
configure_rio(
    cloud_defaults=True,
    aws={"aws_unsigned": True},
    AWS_S3_ENDPOINT="s3.af-south-1.amazonaws.com",
)

Next, we open the Digital Earth Africa STAC catalogue, which provides a structured index over the underlying S3 buckets.
We then define:
1. spatial bounding box (in latitude/longitude),
2. temporal window, and
3. the Sentinel-3 SLSTR Level-2 LST collection to query:

In [4]:
# Open the stac catalogue
catalog = Client.open("https://explorer.digitalearth.africa/stac")

# Set a bounding box
# [xmin, ymin, xmax, ymax] in latitude and longitude
bbox = [-19.6, -17.8, 35.5,  37.3]
((np.float64(-19.6), np.float64(-17.8)), (np.float64(35.5), np.float64(37.3)))

# Set a start and end date
start_date = '2026-05-01'
end_date =  '2026-05-30'

To load other Sentinel-3 SLSTR Level-2 LST product, the user only needs to change the value inside the collections list. The product is exposed as a separate STAC collection 

The available options are:
 - `s3_slstr_l2_lst` to work with the product.
   
To load other Sentinel-3 SLSTR Level-2 LST product, use `s3_slstr_l2_lst` in the collections list with the collection corresponding to their variable of interest.

In [5]:
# Set the STAC collections
collections = ["s3_slstr_l2_lst"]

The code block below builds and executes a STAC search query using the parameters you defined earlier, then reports how many matching datasets were found.

First, a query is constructed against the Digital Earth Africa STAC catalogue using:
 - the spatial extent (bbox)
 - the selected Sentinel-3 SLSTR Level-2 LST collection(s) (collections)
 - the time range (start_date to end_date)

In [6]:
query = catalog.search(
    bbox=bbox, collections=collections, datetime=f"{start_date}/{end_date}"
)

Next, the query is executed and all matching STAC items (each representing an individual dataset, typically a single overpass or time slice) are retrieved:

In [ ]:
# Search the STAC catalog for all items matching the query
items = list(query.items())

Finally, the total number of datasets returned by the search is printed:

In [ ]:
print(f"Found: {len(items):d} datasets")

#### Loads the STAC items into an Xarray Dataset

This step **loads the STAC items into an Xarray Dataset** using cloud-native access, converting the Sentinel-3 SLSTR Level-2 LST STAC assets into an analysis-ready data cube.

Here is what each parameter in `stac_load` is doing:

##### Explanation of key parameters

* **`items`**
  The list of STAC items returned from the catalogue search. Each item corresponds to a  Sentinel-3 SLSTR Level-2 LST observation.

* **`crs="EPSG:4326"`**
  Reprojects the data to geographic coordinates (latitude/longitude), making it easy to work with vector boundaries and other global datasets.

* **`chunks={}`**
  Loads the data lazily (Dask-ready) without forcing a specific chunk size. This is appropriate for small time ranges or exploratory analysis.

* **`groupby="solar_day"`**
  Groups individual overpasses by **solar day**, which is particularly important for Sentinel-3 LST since multiple orbits can occur within a single day.

* **`bbox=bbox`**
  Spatially subsets the data to the defined area of interest, reducing data volume and improving performance.

* **`resolution=0.01`**
  Sets the output spatial resolution (~1000m at the equator). The negative value ensures correct north-up orientation.

In [ ]:
ds_S3 = stac_load(
    items,
    crs='EPSG:4326',
    chunks={},
    groupby="solar_day",
    bbox=bbox,
    resolution = -0.01,
)

#### Resulting dataset

The output, `ds_S3`, is an **Xarray Dataset** that:

* contains gridded Sentinel-3 SLSTR Level-2 LST
* is spatially and temporally subsetted
* is ready for time-series analysis, mapping, or aggregation
* streams data directly from Cloud-Optimised GeoTIFFs in S3

Displaying `ds_S3` in the notebook allows you to inspect:

* available variables
* spatial dimensions (`x`, `y`)
* temporal dimension (`time` or `solar_day`)
* metadata and coordinate information

This dataset can now be used for visualisation, statistics, or further cloud-scale processing.

In [ ]:
# View the Xarray Dataset
ds_S3

#### Applying Scale Factors and Offsets to Sentinel-3 OLCI Measurements

Many measurements in the Sentinel-3 OLCI Level-2 Water Full Resolution product are stored as scaled integers to reduce file size while retaining the required measurement precision. To convert a stored integer value into its physical value, apply the scale_factor and add_offset attributes provided in the [variable metadata](https://docs.digitalearthafrica.org/en/latest/sandbox/notebooks/Datasets/Sentinel_3_Water.html#Applying-Scale-Factors-and-Offsets-to-Sentinel-3-OLCI-Measurements). The scale_factor changes the magnitude of the stored value, while the add_offset shifts it to the correct physical range.

#### Masking
This line applies the quality mask to the dataset, ensuring that only valid Sentinel-3 OLCI L2 WATER observations are retained for analysis. This is a recommended best practice step before any aggregation, mapping, or interpretation of the products.

In [ ]:
# Select a single observation for comparison
time_idx = 8

# Unmasked and apply the scale factor
b03 = (ds_S3["TSM_NN"].isel(time=time_idx)) * 0.0001

# Masked and apply the scale factor
b03_masked = (ds_S3["TSM_NN"].where(ds_S3.dataMask == 1).isel(time=time_idx)) * 0.0001

# Plot comparison
fig, ax = plt.subplots(1, 2, figsize=(12,5))

b03.plot(ax=ax[0], robust=True)
ax[0].set_title("Without dataMask")

b03_masked.plot(ax=ax[1], robust=True)
ax[1].set_title("With dataMask")

plt.tight_layout()

### Visual Insights from the Dataset

The plot below provides a **clear visual summary of what this dataset offers and why it is powerful**:

In [ ]:
#mask and apply the scale factor
(ds_S3['TSM_NN'].where(ds_S3.dataMask == 1).isel(time=[0, 8, 18, 24])* 0.0001).plot(robust=True, col="time", col_wrap=4);

#### What this visual shows

* Each panel represents **total_suspended_matter for a single solar day**, gridded over the area of interest.
* The use of `col="time"` creates a **small-multiples view**, making it easy to compare spatial patterns from day to day.


#### Why this is informative

* You can immediately see **hotspots and gradients** in total_suspended_matter across space.
* Day-to-day differences highlight **temporal variability**, which may be linked to:
* Masked (blank) areas clearly show where **quality filtering** has removed invalid pixels, reinforcing confidence in the data.

#### Why this should excite users

* This is **satellite data streamed directly from the cloud**, not pre-downloaded files.
* With just a few lines of code, users can:

  * visualise total_suspended_matter over any region
  * build time series
* The same workflow scales seamlessly from **two days** to **multiple years** of observations.

#### Takeaway

This visual demonstrates that the dataset is:

* **analysis-ready**
* **spatially rich**
* **temporally dense**
* and perfectly suited for **exploratory science, policy analysis, and storytelling**


### Limitations and next steps

*Are there persistent total_suspended_matter in specific regions that recur seasonally, and how do they relate to land use, climate variability, and human activity?*

**Why these data are suitable:**
Sentinel-3 OLCI L2 WATER provides near-daily, continental-scale coverage, making it possible to detect recurring spatial patterns and seasonal signals that are difficult to observe with sparse ground measurements.

**Recommendations for answering this question:**

* Analyse **multi-year time series** rather than short windows to separate seasonal cycles from noise.
* Apply **strict quality and cloud filtering** before any aggregation.
* Use **consistent spatial grids and temporal grouping** (e.g. solar day or monthly composites).
* Validate patterns where possible using inventories or independent observations.

This approach allows users to move from short-term exploration to **robust, policy-relevant insights**.
